In [9]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

openai.api_key  = os.environ['OPENROUTER_API_KEY']

In [10]:

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
  api_key=os.environ['OPENROUTER_API_KEY'],
  base_url="https://openrouter.ai/api/v1",
  model="openrouter/free",
  temperature=0
)

# 1.Data Loading

In [11]:
from langchain_community.document_loaders import PyPDFLoader
# making the file path relative to the current file
Data_path = "../data/HamzaEldafrawy.pdf"
loader = PyPDFLoader(Data_path)
pages = loader.load()

In [12]:
len(pages)

2

In [13]:
print(pages[0].page_content)

Hamza Basem Mohamed Ahmed Eldafrawy
Cairo, Egypt — 01140728368 — hamza.eldefrawy4@gmail.com
LinkedIn — GitHub
PROFILE
Computer and Systems Engineering student (Class of 2027, GPA: 3.62) with experience in backend development, distributed
systems, machine learning, and data-driven architectures. Built foundational projects involving socket programming, recom-
mendation systems, and software architecture using Python, C#, Java, and C++. Dedicated to a first-principles engineering
mindset, combining robust backend engineering with intelligent systems to build scalable, real-world applications.
EDUCATION
Ain Shams University — Faculty of EngineeringCairo, Egypt
B.Sc. in Computer and Systems EngineeringExpected Graduation: 2027
•Cumulative GPA:3.62/4.00
•Relevant Coursework:Deep Learning, Data Structures & Algorithms, Operating Systems, Parallel and Distributed Systems,
Databases, Fundamentals of Big Data Analysis.
TECHNICAL SKILLS
•Programming Languages:Python, C#, SQL, C++, C, Java
•Data 

# 2.Chunking

In [14]:
from langchain_classic.text_splitter import TokenTextSplitter
splitter = TokenTextSplitter(
  chunk_size = 200,
  chunk_overlap = 20
)


In [15]:
chunks = splitter.split_documents(pages)

# 3.Embedding and Vector DB

In [16]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-MiniLM-L6-v2",
  model_kwargs = {"device":"cpu"}
)
vectors = embeddings.embed_documents([i.page_content for i in chunks])
# vector store
from langchain_classic.vectorstores import FAISS
vectorDB = FAISS.from_documents(chunks,embeddings)

C:\Users\Hamza Eldafrawy\AppData\Local\Temp\ipykernel_28052\2271255690.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Hamza Eldafrawy\Desktop\Cellula_Intern\personal-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3339.13it/s]


# 4.Retrieval

In [17]:
query = "What is the personal information of Hamza Eldafrawy?"
similar_docs = vectorDB.similarity_search_with_score(query,k=3)
for i in similar_docs :
  print(i[0].page_content)

Hamza Basem Mohamed Ahmed Eldafrawy
Cairo, Egypt — 01140728368 — hamza.eldefrawy4@gmail.com
LinkedIn — GitHub
PROFILE
Computer and Systems Engineering student (Class of 2027, GPA: 3.62) with experience in backend development, distributed
systems, machine learning, and data-driven architectures. Built foundational projects involving socket programming, recom-
mendation systems, and software architecture using Python, C#, Java, and C++. Dedicated to a first-principles engineering
mindset, combining robust backend engineering with intelligent systems to build scalable, real-world applications.
EDUCATION
Ain Shams University — Faculty of EngineeringCairo, Egypt
B.Sc. in Computer and Systems EngineeringExpected Graduation: 2027
•Cumulative GPA:3.62/4.00
•Relevant Coursework:Deep Learning,

•Coordinated communication and academic support within the section across multiple semesters.
•Assisted colleagues with coursework-related challenges, reports, and scheduling conflicts.
•Helped organize c

# 5.Generation

In [18]:
from langchain_classic.prompts import PromptTemplate

text = """
You are a helpful assistant that answers questions about a person.
If you don't know the answer, say you don't know.
Keep your answer within three sentences.Keep the answer as concise as possible.
Use the following pieces of context to answer the question at the end. 
Always say "Any further questions?" at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
template = PromptTemplate.from_template(text)

In [19]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

chat = ConversationalRetrievalChain.from_llm(
  llm = llm,
  retriever = vectorDB.as_retriever(),
  return_source_documents = True,
  combine_docs_chain_kwargs={"prompt": template},
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=False, output_key="answer")
)

C:\Users\Hamza Eldafrawy\AppData\Local\Temp\ipykernel_28052\3881909334.py:9: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=False, output_key="answer")
c:\Users\Hamza Eldafrawy\Desktop\Cellula_Intern\personal-RAG\venv\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self =

In [20]:
response = chat.invoke({
    "question": "I want to know what does Hamza specialize in?",
})

print(response["answer"])
# print(response["source_documents"])


Hamza specializes in backend development, distributed systems, machine learning, and data-driven architectures, with proficiency in Python, C#, Java, and C++. He has completed a Machine Learning Specialization from DeepLearning.AI and a Data Science Track from ALX Africa, focusing on deep learning, classical ML pipelines, and relational database design. His projects include Unix shell implementation and an online language teaching system using OOP principles. Any further questions?
